# Stage 6: Web Demo (FastAPI REST API & Gradio Interface)

This notebook demonstrates:
1. Querying the FastAPI backend programmatically via `requests` / `TestClient`
2. Testing `POST /api/predict` with and without Grad-CAM explanation
3. Inspecting the JSON response and rendering the base64 Grad-CAM overlay
4. Launching the Gradio web app for Hugging Face Spaces deployment

All logic is decoupled into `src/catdog/api.py` and `app_gradio.py`.

In [ ]:
import sys
sys.path.insert(0, '../src')
sys.path.insert(0, '..')

from fastapi.testclient import TestClient
from catdog.api import app
from catdog.config import Config
import base64
import io
from PIL import Image
import matplotlib.pyplot as plt

client = TestClient(app)
config = Config()

## 1. System Health & Active Model Metadata

In [ ]:
resp = client.get('/api/health')
print('Health Response:', resp.json())

## 2. Programmatic Prediction with Grad-CAM (`POST /api/predict`)

In [ ]:
test_image_path = config.test_dir / '1.jpg'
with open(test_image_path, 'rb') as f:
    files = {'file': ('1.jpg', f.read(), 'image/jpeg')}

response = client.post('/api/predict?explain=true', files=files)
data = response.json()

print(f"File:        {data['filename']}")
print(f"Prediction:  {data['label']} ({data['confidence_pct']}% confidence)")
print(f"P(Dog):      {data['dog_probability']:.4f}")
print(f"P(Cat):      {data['cat_probability']:.4f}")
print(f"Latency:     {data['inference_time_ms']:.1f} ms")

# Decode and display the returned Grad-CAM heatmap overlay
if data['gradcam_base64']:
    header, encoded = data['gradcam_base64'].split(',', 1)
    image_data = base64.b64decode(encoded)
    overlay_img = Image.open(io.BytesIO(image_data))
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 4))
    ax1.imshow(Image.open(test_image_path))
    ax1.set_title('Original Input')
    ax1.axis('off')
    
    ax2.imshow(overlay_img)
    ax2.set_title('API Grad-CAM Response')
    ax2.axis('off')
    plt.tight_layout()
    plt.show()

## 3. Launching the Gradio Web App
Run `python app_gradio.py` from the terminal or execute the cell below to launch the interactive UI.

In [ ]:
# from app_gradio import demo
# demo.launch(server_name='127.0.0.1', server_port=7860)